In [6]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import implicit
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [7]:
train = pd.read_parquet("../data/processed/train.parquet")
test = pd.read_parquet("../data/processed/test.parquet")
item_features = pd.read_parquet("../data/processed/item_features.parquet")

train.shape, test.shape

((495761, 5), (74156, 5))

In [8]:
user_enc = LabelEncoder()
item_enc = LabelEncoder()

all_users = pd.concat([train["customer_id"], test["customer_id"]]).unique()
all_items = pd.concat([train["article_id"], test["article_id"]]).unique()

user_enc.fit(all_users)
item_enc.fit(all_items)

train["user_idx"] = user_enc.transform(train["customer_id"])
train["item_idx"] = item_enc.transform(train["article_id"])
test["user_idx"] = user_enc.transform(test["customer_id"])
test["item_idx"] = item_enc.transform(test["article_id"])

n_users = len(user_enc.classes_)
n_items = len(item_enc.classes_)

pd.DataFrame({
    "항목": ["전체 유저 수", "전체 상품 수"],
    "값": [n_users, n_items],
})

,항목,값
0,전체 유저 수,50000
1,전체 상품 수,32806


In [9]:
user_id_to_idx = dict(zip(user_enc.classes_, range(n_users)))
item_idx_to_id = dict(zip(range(n_items), item_enc.classes_))

joblib.dump(user_id_to_idx, "../backend/models/user_idx.pkl")
joblib.dump(item_idx_to_id, "../backend/models/item_idx.pkl")

['../backend/models/item_idx.pkl']

### Sparse Matrix 생성

In [10]:
interaction_counts = (
    train.groupby(["user_idx", "item_idx"])["article_id"]
    .count()
    .reset_index()
    .rename(columns={"article_id": "count"})
)

user_item_matrix = sp.csr_matrix(
    (
        interaction_counts["count"].values,
        (interaction_counts["user_idx"].values, interaction_counts["item_idx"].values),
    ),
    shape=(n_users, n_items),
)

item_user_matrix = user_item_matrix.T.tocsr()

joblib.dump(user_item_matrix, "../backend/models/user_item_sparse.pkl")

user_item_matrix.shape, user_item_matrix.nnz

((50000, 32806), 430922)

### ALS Model 학습

In [11]:
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.1,
    random_state=42,
)

model.fit(item_user_matrix)

c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [12]:
joblib.dump(model, "../backend/models/als_model.pkl")

['../backend/models/als_model.pkl']

### 평가 함수

In [13]:
def get_recommendations(model, user_item_matrix, user_idx, n=12):
    ids, scores = model.recommend(
        user_idx,
        user_item_matrix[user_idx],
        N=n,
        filter_already_liked_items=True,
    )
    return ids, scores

def hit_rate_at_k(model, user_item_matrix, test_df, k=12, sample_n=2000):
    max_user_idx = model.user_factors.shape[0]
    valid_users = test_df[test_df["user_idx"] < max_user_idx]["user_idx"].unique()

    np.random.seed(42)
    sampled = np.random.choice(valid_users, size=min(sample_n, len(valid_users)), replace=False)

    hits = 0
    for u in sampled:
        actual = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not actual:
            continue
        rec_ids, _ = get_recommendations(model, user_item_matrix, u, n=k)
        if len(set(rec_ids) & actual) > 0:
            hits += 1

    return hits / len(sampled)

def ndcg_at_k(model, user_item_matrix, test_df, k=12, sample_n=2000):
    max_user_idx = model.user_factors.shape[0]
    valid_users = test_df[test_df["user_idx"] < max_user_idx]["user_idx"].unique()

    np.random.seed(42)
    sampled = np.random.choice(valid_users, size=min(sample_n, len(valid_users)), replace=False)

    ndcg_scores = []
    for u in sampled:
        actual = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not actual:
            continue
        rec_ids, _ = get_recommendations(model, user_item_matrix, u, n=k)
        dcg = sum(
            1 / np.log2(rank + 2)
            for rank, item in enumerate(rec_ids)
            if item in actual
        )
        idcg = sum(1 / np.log2(rank + 2) for rank in range(min(len(actual), k)))
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(ndcg_scores)



In [14]:
results = []

for factors in [32, 64, 128]:
    m = implicit.als.AlternatingLeastSquares(
        factors=factors,
        iterations=20,
        regularization=0.1,
        random_state=42,
    )
    m.fit(item_user_matrix)
    hr = hit_rate_at_k(m, user_item_matrix, test, k=12, sample_n=1000)
    nd = ndcg_at_k(m, user_item_matrix, test, k=12, sample_n=1000)
    results.append({
        "factors": factors,
        "Hit Rate@12": round(hr, 4),
        "NDCG@12": round(nd, 4),
    })

pd.DataFrame(results)

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

,factors,Hit Rate@12,NDCG@12
0,32,0.000,0.0000
1,64,0.001,0.0001
2,128,0.000,0.0000


### 평가 실행

In [15]:
hit_rate = hit_rate_at_k(model, user_item_matrix, test, k=12, sample_n=2000)
ndcg = ndcg_at_k(model, user_item_matrix, test, k=12, sample_n=2000)

pd.DataFrame({
    "지표": ["Hit Rate@12", "NDCG@12"],
    "ALS (factors=64)": [round(hit_rate, 4), round(ndcg, 4)],
})

,지표,ALS (factors=64)
0,Hit Rate@12,0.0010
1,NDCG@12,0.0002


In [17]:
sample_user_id = train["customer_id"].iloc[0]
sample_user_idx = user_id_to_idx[sample_user_id]

rec_ids, rec_scores = get_recommendations(model, user_item_matrix, sample_user_idx, n=12)

valid = [(i, s) for i, s in zip(rec_ids, rec_scores) if i in item_idx_to_id]
rec_article_ids = [item_idx_to_id[i] for i, _ in valid]
rec_scores_valid = [s for _, s in valid]

user_history = (
    train[train["customer_id"] == sample_user_id]
    .merge(
        item_features[["article_id", "product_type_name", "colour_group_name"]],
        on="article_id",
        how="left",
    )
    [["article_id", "product_type_name", "colour_group_name"]]
    .drop_duplicates()
)

rec_result = (
    pd.DataFrame({"article_id": rec_article_ids, "score": rec_scores_valid})
    .merge(
        item_features[["article_id", "product_type_name", "colour_group_name"]],
        on="article_id",
        how="left",
    )
)

user_history

,article_id,product_type_name,colour_group_name
0,0762846007,Shirt,Light Beige
1,0762846008,Shirt,Light Pink


In [18]:
rec_result

,article_id,score,product_type_name,colour_group_name
0,0585158008,0.201505,Swimwear bottom,Yellowish Brown
1,0800768003,0.128571,Top,Yellow
2,0811660002,0.103904,Sweater,Light Green
3,0699361004,0.098665,Underwear bottom,Dark Red
4,0909831001,0.084429,Dress,Dark Blue
5,0720810011,0.080740,Trousers,Black
6,0762796007,0.078780,Sweater,Dark Blue
7,0278811011,0.077268,Underwear bottom,Beige


In [20]:
max_user_idx = model.user_factors.shape[0]
all_user_recs = {}

for uid in train["customer_id"].unique():
    uidx = user_id_to_idx.get(uid)
    if uidx is None or uidx >= max_user_idx:
        continue
    rec_ids, _ = get_recommendations(model, user_item_matrix, uidx, n=12)
    valid_ids = [item_idx_to_id[i] for i in rec_ids if i in item_idx_to_id]
    if valid_ids:
        all_user_recs[uid] = valid_ids

joblib.dump(all_user_recs, "../backend/models/als_recommendations.pkl")
len(all_user_recs)

30523